In [1]:
import torch
print(torch.cuda.is_available())       # must be True
print(torch.cuda.get_device_name(0))   # your GPU name

True
NVIDIA GeForce GTX 1650


In [9]:
import cv2
import os
import pandas as pd
import pickle

### Ideas to try out:

 - Histogram equalization, a.k.a. CLAHE (with opencv)
 - Noise Reduction with Gaussian or BLUR filters
 - Data augmentation with Albumentation so that sample count increases
 - Contrast Enhancement (wiht opencv)

### After preprocessing steps above, we may proceed to use models like YOLO, CNNs, DETR to predict the diseases

 - YOLO
 - DETR
 - CNN
 - Transformers
 - CNN + NN
 - CNN + Transformers (simialar to DETR)

In [10]:
# when you need to read the image, use "train/"+trains[x], when you need the metadata of image, use images[x]

trains = os.listdir("train/")
images = [im.split(".")[0] for im in trains]

with open("image_names.pickle", "wb") as f:
    pickle.dump(images, f)

In [11]:
a_image = cv2.imread("train/"+trains[0])
print("Shape: ", a_image.shape)
print("Size: ", a_image.size)
print("Data type: ", a_image.dtype)

Shape:  (1024, 1024, 3)
Size:  3145728
Data type:  uint8


In [12]:
train_metadata = pd.read_csv("train.csv")
train_metadata[train_metadata["image_id"] == images[0]]

,image_id,class_name,class_id,rad_id,x_min,y_min,x_max,y_max
14880,00JgsY3R0C6VQrT7VDFcoqW2J7dOfULr,No finding,14,R11,NaN,NaN,NaN,NaN
15183,00JgsY3R0C6VQrT7VDFcoqW2J7dOfULr,No finding,14,R1,NaN,NaN,NaN,NaN
26796,00JgsY3R0C6VQrT7VDFcoqW2J7dOfULr,No finding,14,R5,NaN,NaN,NaN,NaN


In [13]:
# Compute the new 1024 bounded disease pixel locations

new_size = 1024
df = train_metadata.copy()
img_size_df = pd.read_csv('img_size.csv').set_index('image_id')

def compute_new_coords(row):
    img_id = str(row['image_id']).split(".")[0]
    orig_h = img_size_df.loc[img_id, 'dim0']
    orig_w = img_size_df.loc[img_id, 'dim1']
    
    cx = ((row['x_min'] + row['x_max']) / 2) / orig_w
    cy = ((row['y_min'] + row['y_max']) / 2) / orig_h
    w  = (row['x_max'] - row['x_min']) / orig_w
    h  = (row['y_max'] - row['y_min']) / orig_h
    
    return pd.Series({
        'x_min_new': ((cx - w/2) * new_size),
        'y_min_new': ((cy - h/2) * new_size),
        'x_max_new': ((cx + w/2) * new_size),
        'y_max_new': ((cy + h/2) * new_size),
    })

df[['x_min_new', 'y_min_new', 'x_max_new', 'y_max_new']] = df.apply(compute_new_coords, axis=1)

cols = ['x_min_new', 'y_min_new', 'x_max_new', 'y_max_new']
mask = df['class_id'] != 14
df.loc[mask, cols] = df.loc[mask, cols].astype('int16')

df.to_csv('train_metadata_1024.csv', index=False)

In [14]:
t_mt_1024 = pd.read_csv("train_metadata_1024.csv")
t_mt_1024.shape

(45925, 12)

In [15]:
t_mt_1024[t_mt_1024["image_id"] == images[0]]

,image_id,class_name,class_id,rad_id,x_min,y_min,x_max,y_max,x_min_new,y_min_new,x_max_new,y_max_new
14880,00JgsY3R0C6VQrT7VDFcoqW2J7dOfULr,No finding,14,R11,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
15183,00JgsY3R0C6VQrT7VDFcoqW2J7dOfULr,No finding,14,R1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
26796,00JgsY3R0C6VQrT7VDFcoqW2J7dOfULr,No finding,14,R5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [16]:
import torch
print(torch.cuda.is_available())   # True
print(torch.version.cuda)          # 12.1 — that's fine

True
12.4
